# RPM validation for all data files

This notebook scans every CSV in `../data`, groups recordings into **healthy**, **misalignment**, **rear_ball**, and **front_ball**, and calculates actual RPM from the `pg_rpm` pulse signal.

The calculation follows `RPM_calculation.ipynb`: one mechanical revolution equals 6 rising PG edges. A file passes when actual RPM differs from the filename setpoint by no more than `max(25 RPM, 5%)`. Failed files are marked with a red **X**.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import HTML, display

PULSES_PER_REV = 6
CHUNK_SIZE = 500_000
RPM_TOLERANCE_PERCENT = 5.0
RPM_TOLERANCE_MIN = 25.0
GROUPS = ("healthy", "misalignment", "rear_ball", "front_ball")

data_candidates = (Path.cwd() / "data", Path.cwd().parent / "data")
DATA_DIR = next((path.resolve() for path in data_candidates if path.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find the repository data directory.")

name_pattern = re.compile(
    r"^analize_(?P<group>healthy|misalignment|rear_ball|front_ball).*?_(?P<target_rpm>\d+)rpm_"
)

def calculate_actual_rpm(csv_path):
    """Calculate revolution-averaged RPM from six PG rising edges."""
    pulse_time_chunks = []
    previous_pg = None

    for chunk in pd.read_csv(
        csv_path,
        usecols=["t_us", "pg_rpm"],
        chunksize=CHUNK_SIZE,
    ):
        times = pd.to_numeric(chunk["t_us"], errors="coerce").to_numpy(dtype=float)
        pg = pd.to_numeric(chunk["pg_rpm"], errors="coerce").to_numpy(dtype=float)
        rising = np.zeros(len(chunk), dtype=bool)

        if len(chunk):
            if previous_pg is not None:
                rising[0] = previous_pg == 0 and pg[0] == 1
            rising[1:] = (pg[:-1] == 0) & (pg[1:] == 1)
            valid_edges = rising & np.isfinite(times)
            pulse_time_chunks.append(times[valid_edges])
            finite_pg = pg[np.isfinite(pg)]
            if finite_pg.size:
                previous_pg = finite_pg[-1]

    if not pulse_time_chunks:
        raise ValueError("No PG data found")

    pulse_times_us = np.concatenate(pulse_time_chunks)
    if pulse_times_us.size <= PULSES_PER_REV:
        raise ValueError("Not enough PG rising edges")

    revolution_us = (
        pulse_times_us[PULSES_PER_REV:] - pulse_times_us[:-PULSES_PER_REV]
    )
    rpm_values = 60_000_000.0 / revolution_us
    rpm_values = rpm_values[np.isfinite(rpm_values) & (rpm_values > 0)]
    if not rpm_values.size:
        raise ValueError("No valid revolution periods")

    return {
        "actual_rpm": rpm_values.mean(),
        "rpm_std": rpm_values.std(),
        "rpm_min": rpm_values.min(),
        "rpm_max": rpm_values.max(),
        "pulses": pulse_times_us.size,
    }

print(f"Data directory: {DATA_DIR}")
print("RPM calculation functions are ready.")

Data directory: C:\Asmeninis\OpenBLDCData\data
RPM calculation functions are ready.


In [3]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
matched_files = []
for csv_path in csv_files:
    match = name_pattern.match(csv_path.name)
    if match:
        matched_files.append((csv_path, match))

print(
    f"Found {len(csv_files)} CSV files: {len(matched_files)} grouped files and "
    f"{len(csv_files) - len(matched_files)} ungrouped files."
)

rows = []
for index, (csv_path, match) in enumerate(matched_files, start=1):
    group = match.group("group")
    target_rpm = float(match.group("target_rpm"))
    tolerance_rpm = max(RPM_TOLERANCE_MIN, target_rpm * RPM_TOLERANCE_PERCENT / 100)

    row = {
        "group": group,
        "file": csv_path.name,
        "target_rpm": target_rpm,
        "tolerance_rpm": tolerance_rpm,
    }

    try:
        rpm_stats = calculate_actual_rpm(csv_path)
        difference_rpm = rpm_stats["actual_rpm"] - target_rpm
        matches = abs(difference_rpm) <= tolerance_rpm
        row.update(
            **rpm_stats,
            difference_rpm=difference_rpm,
            matches=matches,
            status="OK" if matches else "X",
            error="",
        )
    except Exception as exc:
        row.update(
            actual_rpm=np.nan,
            rpm_std=np.nan,
            rpm_min=np.nan,
            rpm_max=np.nan,
            pulses=0,
            difference_rpm=np.nan,
            matches=False,
            status="X",
            error=str(exc),
        )

    rows.append(row)
    print(f"\rProcessed {index}/{len(matched_files)} files", end="")

print()
results = pd.DataFrame(rows)
results["group"] = pd.Categorical(results["group"], categories=GROUPS, ordered=True)
results = results.sort_values(["group", "target_rpm", "file"]).reset_index(drop=True)

summary = (
    results.groupby("group", observed=True)
    .agg(files=("file", "size"), matching=("matches", "sum"))
    .reset_index()
)
summary["with_red_x"] = summary["files"] - summary["matching"]

display(HTML("<h2>Validation summary</h2>"))
display(summary.style.hide(axis="index"))

table_columns = [
    "file",
    "target_rpm",
    "actual_rpm",
    "difference_rpm",
    "tolerance_rpm",
    "rpm_std",
    "rpm_min",
    "rpm_max",
    "pulses",
    "status",
    "error",
]

def status_style(value):
    if value == "X":
        return "color: #c62828; font-weight: 800; font-size: 1.15em"
    return "color: #1b5e20; font-weight: 700"

for group in GROUPS:
    group_results = results.loc[results["group"] == group, table_columns]
    display(HTML(f"<h2>{group} ({len(group_results)} files)</h2>"))
    styled = (
        group_results.style
        .hide(axis="index")
        .format(
            {
                "target_rpm": "{:.0f}",
                "actual_rpm": "{:.1f}",
                "difference_rpm": "{:+.1f}",
                "tolerance_rpm": "{:.1f}",
                "rpm_std": "{:.1f}",
                "rpm_min": "{:.1f}",
                "rpm_max": "{:.1f}",
                "pulses": "{:.0f}",
            },
            na_rep="-",
        )
        .map(status_style, subset=["status"])
    )
    display(styled)

Found 180 CSV files: 179 grouped files and 1 ungrouped files.
Processed 179/179 files


group,files,matching,with_red_x
healthy,71,70,1
misalignment,65,64,1
rear_ball,21,21,0
front_ball,22,21,1


file,target_rpm,actual_rpm,difference_rpm,tolerance_rpm,rpm_std,rpm_min,rpm_max,pulses,status,error
analize_healthy10_500rpm_128mA_bat.csv,500,504.7,+4.7,25.0,1.5,499.9,508.7,14240,OK,
analize_healthy10_500rpm_19mA_bat.csv,500,515.4,+15.4,25.0,2.2,512.5,517.5,13570,OK,
analize_healthy10_500rpm_38mA_bat.csv,500,512.3,+12.3,25.0,1.6,508.2,517.4,13539,OK,
analize_healthy10_500rpm_64mA_bat.csv,500,503.9,+3.9,25.0,1.3,499.8,508.6,14238,OK,
analize_healthy10_500rpm_96mA_bat.csv,500,505.6,+5.6,25.0,2.0,503.9,508.7,13313,OK,
analize_healthy10_500rpm_96mA_mait.csv,500,497.6,-2.4,25.0,2.0,495.6,500.2,16673,OK,
analize_healthy10_ENV_500rpm_96mA_bat.csv,500,512.0,+12.0,25.0,1.7,508.2,517.2,22291,OK,
analize_healthy20_500rpm_128mA_mait.csv,500,504.3,+4.3,25.0,0.8,500.0,508.5,14040,OK,
analize_healthy20_500rpm_19mA_mait.csv,500,512.6,+12.6,25.0,1.0,508.4,517.2,13442,OK,
analize_healthy20_500rpm_38mA_mait.csv,500,511.2,+11.2,25.0,2.1,508.3,512.9,13472,OK,


file,target_rpm,actual_rpm,difference_rpm,tolerance_rpm,rpm_std,rpm_min,rpm_max,pulses,status,error
analize_misalignment10_500rpm_128mA_mait.csv,500,505.8,+5.8,25.0,2.1,504.1,508.6,4038,OK,
analize_misalignment10_500rpm_19mA_bat.csv,500,-,-,25.0,-,-,-,0,X,Not enough PG rising edges
analize_misalignment10_500rpm_19mA_mait.csv,500,500.1,+0.1,25.0,0.8,495.8,504.3,3943,OK,
analize_misalignment10_500rpm_38mA_mait.csv,500,508.2,+8.2,25.0,1.1,504.2,508.6,4008,OK,
analize_misalignment10_500rpm_64mA_mait.csv,500,510.8,+10.8,25.0,2.2,508.4,512.9,4139,OK,
analize_misalignment10_500rpm_96mA_bat.csv,500,513.3,+13.3,25.0,1.5,508.5,517.4,13465,OK,
analize_misalignment10_500rpm_96mA_mait.csv,500,509.2,+9.2,25.0,1.6,508.4,512.9,4020,OK,
analize_misalignment10_ENV_500rpm_128mA_mait.csv,500,505.9,+5.9,25.0,2.1,504.1,508.6,3788,OK,
analize_misalignment20_500rpm_128mA_mait.csv,500,506.2,+6.2,25.0,2.1,504.1,508.6,3993,OK,
analize_misalignment20_500rpm_19mA_mait.csv,500,509.3,+9.3,25.0,1.7,508.3,513.0,3975,OK,


file,target_rpm,actual_rpm,difference_rpm,tolerance_rpm,rpm_std,rpm_min,rpm_max,pulses,status,error
analize_rear_ball30_500rpm_128mA_mait.csv,500,498.0,-2.0,25.0,2.1,495.7,500.1,4131,OK,
analize_rear_ball30_500rpm_19mA_mait.csv,500,503.7,+3.7,25.0,2.3,499.9,508.6,5238,OK,
analize_rear_ball30_500rpm_38mA_mait.csv,500,501.1,+1.1,25.0,2.2,495.8,504.3,4558,OK,
analize_rear_ball30_500rpm_64mA_mait.csv,500,507.5,+7.5,25.0,2.1,504.1,512.9,4000,OK,
analize_rear_ball30_500rpm_96mA_mait.csv,500,511.3,+11.3,25.0,2.1,508.4,512.9,5325,OK,
analize_rear_ball30_1000rpm_128mA_mait.csv,1000,1002.0,+2.0,50.0,5.4,999.6,1017.4,7906,OK,
analize_rear_ball30_1000rpm_19mA_mait.csv,1000,1007.7,+7.7,50.0,8.4,999.5,1017.5,8039,OK,
analize_rear_ball30_1000rpm_38mA_mait.csv,1000,1003.8,+3.8,50.0,7.0,999.6,1017.2,7916,OK,
analize_rear_ball30_1000rpm_64mA_mait.csv,1000,1011.9,+11.9,50.0,7.7,999.5,1017.4,8401,OK,
analize_rear_ball30_1000rpm_96mA_mait.csv,1000,1005.7,+5.7,50.0,8.0,999.6,1017.4,8645,OK,


file,target_rpm,actual_rpm,difference_rpm,tolerance_rpm,rpm_std,rpm_min,rpm_max,pulses,status,error
analize_front_ball30_500rpm_128mA_mait.csv,500,507.5,+7.5,25.0,5.3,491.8,517.3,4053,OK,
analize_front_ball30_500rpm_19mA_mait.csv,500,533.0,+33.0,25.0,6.2,495.9,545.5,4314,X,
analize_front_ball30_500rpm_38mA_mait.csv,500,519.2,+19.2,25.0,5.4,491.8,531.0,4356,OK,
analize_front_ball30_500rpm_64mA_mait.csv,500,511.9,+11.9,25.0,6.4,483.9,526.3,5132,OK,
analize_front_ball30_500rpm_96mA_mait.csv,500,504.3,+4.3,25.0,5.5,483.9,517.3,4027,OK,
analize_front_ball30_1000rpm_128mA_mait.csv,1000,1018.6,+18.6,50.0,6.5,999.6,1034.7,8132,OK,
analize_front_ball30_1000rpm_19mA_mait.csv,1000,1007.9,+7.9,50.0,8.6,983.4,1017.4,7956,OK,
analize_front_ball30_1000rpm_38mA_mait.csv,1000,1011.5,+11.5,50.0,8.0,983.6,1017.5,7984,OK,
analize_front_ball30_1000rpm_64mA_mait.csv,1000,1016.4,+16.4,50.0,3.6,999.7,1034.5,8334,OK,
analize_front_ball30_1000rpm_96mA_mait.csv,1000,997.5,-2.5,50.0,5.9,983.3,1000.5,8397,OK,
